[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/28_neural_networks/04_regularization_and_training_recipes.ipynb)

# Bài 04 — Regularization và quy trình huấn luyện

**🎯 Mục tiêu**
- Nhận diện overfitting ở mạng nơ-ron; vì sao mạng lớn "thuộc lòng" được nhiễu.
- **Weight decay** (L2, AdamW), **dropout** (from scratch, train vs eval), **nhiễu đầu vào / data augmentation**, **early stopping**, và "regularizer tốt nhất": thêm dữ liệu.
- Quy trình huấn luyện có hệ thống và bảng gỡ lỗi; template model cho mạng feed-forward.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/28_neural_networks

In [ ]:
import copy

import matplotlib.pyplot as plt
import numpy as np

from ml_course.visualization import set_style

try:
    import torch
    from torch import nn
except ImportError as e:
    raise ImportError("Chương này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()

## 1. Overfitting: mạng lớn, ít dữ liệu nhiễu

30 điểm từ $y = \sin x + \varepsilon$, $\varepsilon \sim \mathcal N(0, 0.3^2)$; mạng 1 → 256 → 256 → 1 (~66 000 tham số), Adam 3 000 bước. Mức nhiễu $\sigma^2 = 0.09$ là giới hạn dưới của MSE trên dữ liệu **mới**; MSE train thấp hơn nhiều thì mạng đang khớp cả nhiễu.

In [ ]:
r = np.random.default_rng(0)
x_train = np.sort(r.uniform(-3, 3, 30))
y_train = np.sin(x_train) + r.normal(0, 0.3, 30)
x_val = r.uniform(-3, 3, 30)                                            # tập validation riêng (cho early stopping)
y_val = np.sin(x_val) + r.normal(0, 0.3, 30)
x_grid = np.linspace(-3, 3, 500)
to_t = lambda a: torch.tensor(np.asarray(a)[:, None], dtype=torch.float32)
Xt, Yt, Xv, Yv, Xg = to_t(x_train), to_t(y_train), to_t(x_val), to_t(y_val), to_t(x_grid)


def make_net(p_drop=0.0, seed=0):
    torch.manual_seed(seed)
    layers = [nn.Linear(1, 256), nn.ReLU()]
    if p_drop:
        layers.append(nn.Dropout(p_drop))
    layers += [nn.Linear(256, 256), nn.ReLU()]
    if p_drop:
        layers.append(nn.Dropout(p_drop))
    return nn.Sequential(*layers, nn.Linear(256, 1))


def fit(weight_decay=0.0, p_drop=0.0, input_noise=0.0, steps=3000, seed=0, X=Xt, Y=Yt, track_every=0):
    model = make_net(p_drop, seed)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=weight_decay)
    history = []
    for step in range(steps):
        model.train()
        x_in = X + input_noise * torch.randn_like(X) if input_noise else X
        opt.zero_grad()
        nn.functional.mse_loss(model(x_in), Y).backward()
        opt.step()
        if track_every and step % track_every == 0:
            model.eval()
            with torch.no_grad():
                history.append((step, nn.functional.mse_loss(model(Xv), Yv).item(), copy.deepcopy(model.state_dict())))
    model.eval()
    return model, history


def evaluate(model):
    with torch.no_grad():
        train_mse = nn.functional.mse_loss(model(Xt), Yt).item()
        true_mse = float(np.mean((model(Xg).numpy().ravel() - np.sin(x_grid)) ** 2))   # sai số so với hàm thật
    return train_mse, true_mse


base, _ = fit()
tr, te = evaluate(base)
print(f"không regularization: MSE train = {tr:.4f} (mức nhiễu 0.09) | sai số so với sin(x) = {te:.4f}")

## 2. Weight decay (L2)

Phạt $\frac\lambda2\lVert\theta\rVert^2$ → mỗi bước kéo trọng số về 0 một chút ("decay"). Trọng số nhỏ → hàm **trơn** hơn (đạo hàm theo đầu vào bị chặn bởi tích các chuẩn trọng số). Với Adam dùng **AdamW** (bài 03).

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 4))
for ax, wd in zip(axes, [0.0, 0.1, 1.0, 10.0]):
    model, _ = fit(weight_decay=wd)
    tr, te = evaluate(model)
    w_norm = torch.sqrt(sum((p**2).sum() for n, p in model.named_parameters() if "weight" in n)).item()
    with torch.no_grad():
        ax.plot(x_grid, model(Xg).numpy().ravel(), "r", label="mạng")
    ax.plot(x_grid, np.sin(x_grid), "g--", label="sin(x)")
    ax.scatter(x_train, y_train, s=15, color="k")
    ax.set_title(f"weight decay = {wd}: train {tr:.3f} | sai số thật {te:.3f}")
    print(f"weight decay = {wd:<5}: MSE train = {tr:.4f} | sai số so với sin(x) = {te:.4f} | ‖W‖ = {w_norm:.1f}")
axes[0].legend(fontsize=8)
plt.show()

Weight decay nhỏ (0.1) gần như không đổi gì; đủ lớn (1–10) thì MSE train tăng (không còn thuộc lòng nhiễu), chuẩn trọng số giảm mạnh, đường khớp trơn hơn và sai số so với hàm thật giảm khoảng một nửa. Quá lớn sẽ underfit — chọn $\lambda$ bằng validation (với AdamW, độ lớn hợp lý của `weight_decay` còn phụ thuộc learning rate và số bước).

## 3. Dropout

Khi train, mỗi nơ-ron bị **tắt ngẫu nhiên** với xác suất $p$; các nơ-ron còn lại được nhân $\frac{1}{1-p}$ (**inverted dropout**) để kỳ vọng đầu ra không đổi → khi **eval** không cần làm gì. Trực giác: mạng không thể dựa vào một nơ-ron cụ thể; tương đương huấn luyện **ensemble** của rất nhiều mạng con dùng chung trọng số (Srivastava et al., 2014).

In [ ]:
def dropout_scratch(h, p, training, rng):
    if not training or p == 0:
        return h
    mask = (rng.random(h.shape) >= p) / (1 - p)                         # giữ với xác suất 1 − p, phóng to phần được giữ
    return h * mask


h = np.ones((10000, 4))
out = dropout_scratch(h, 0.5, True, np.random.default_rng(0))
print(f"train: tỉ lệ bị tắt = {np.mean(out == 0):.3f} | trung bình đầu ra = {out.mean():.3f} (kỳ vọng giữ nguyên = 1)")
print(f"eval : đầu ra = đầu vào? {np.array_equal(dropout_scratch(h, 0.5, False, None), h)}")

drop = nn.Dropout(0.5)
x_demo = torch.ones(1, 8)
drop.train()
print("nn.Dropout chế độ train:", drop(x_demo).numpy().ravel())
drop.eval()
print("nn.Dropout chế độ eval :", drop(x_demo).numpy().ravel())

Kết quả của một lần huấn luyện dao động theo seed (khởi tạo, mặt nạ dropout) → lấy trung bình 3 seed:

In [ ]:
for p in [0.0, 0.2, 0.5]:
    results = [evaluate(fit(p_drop=p, seed=sd)[0]) for sd in range(3)]
    tr, te = np.mean(results, axis=0)
    spread = np.std([r[1] for r in results])
    print(f"dropout p = {p}: MSE train TB = {tr:.4f} | sai số so với sin(x) TB = {te:.4f} (± {spread:.4f} giữa các seed)")

Dropout làm MSE train tăng rõ (khó thuộc lòng hơn); sai số so với hàm thật giảm vừa phải và dao động đáng kể giữa các seed. Dropout phát huy mạnh nhất ở mạng lớn, dữ liệu nhiều chiều (ảnh, văn bản); với mạng nhỏ 1 chiều như ở đây, weight decay hoặc nhiễu đầu vào hiệu quả hơn.

## 4. Nhiễu đầu vào và data augmentation

Thêm nhiễu nhỏ vào đầu vào mỗi bước = nói với mạng "các điểm lân cận phải có đầu ra gần nhau" → hàm trơn hơn (về lý thuyết, gần tương đương một dạng regularization Tikhonov — Bishop, 1995). Với ảnh, **data augmentation** (lật, cắt, xoay, đổi màu — chương 29, 33) là phiên bản có hiểu biết miền: tạo thêm dữ liệu mà nhãn **không đổi**.

In [ ]:
for noise in [0.0, 0.1, 0.3]:
    model, _ = fit(input_noise=noise)
    tr, te = evaluate(model)
    print(f"nhiễu đầu vào σ = {noise}: MSE train = {tr:.4f} | sai số so với sin(x) = {te:.4f}")

## 5. Early stopping

Theo dõi loss **validation** trong quá trình train; giữ lại trọng số tốt nhất; dừng khi không cải thiện sau `patience` lần kiểm tra. Rẻ, gần như luôn nên dùng — một dạng regularization ngầm (giới hạn số bước = giới hạn "khoảng cách" trọng số đi được từ điểm khởi tạo nhỏ).

In [ ]:
model, hist = fit(steps=3000, track_every=25)
steps_h = [h[0] for h in hist]
val_h = [h[1] for h in hist]
best_i = int(np.argmin(val_h))
plt.figure(figsize=(9, 4))
plt.plot(steps_h, val_h, label="MSE validation")
plt.axvline(steps_h[best_i], color="r", ls="--", label=f"tốt nhất: bước {steps_h[best_i]}")
plt.xlabel("bước")
plt.legend()
plt.show()
early = make_net()
early.load_state_dict(hist[best_i][2])
early.eval()
print(f"cuối quá trình (bước 3000): sai số so với sin(x) = {evaluate(model)[1]:.4f} | MSE val = {val_h[-1]:.4f}")
print(f"early stopping (bước {steps_h[best_i]}): sai số so với sin(x) = {evaluate(early)[1]:.4f} | MSE val = {val_h[best_i]:.4f}")

## 6. Regularizer tốt nhất: thêm dữ liệu

In [ ]:
r2 = np.random.default_rng(1)
for n in [30, 100, 300]:
    xs = np.sort(r2.uniform(-3, 3, n))
    ys = np.sin(xs) + r2.normal(0, 0.3, n)
    model, _ = fit(X=to_t(xs), Y=to_t(ys))
    with torch.no_grad():
        true_err = float(np.mean((model(Xg).numpy().ravel() - np.sin(x_grid)) ** 2))
    print(f"n = {n:3d} điểm, không regularization: sai số so với sin(x) = {true_err:.4f}")

Cùng mạng, không regularization, nhiều dữ liệu hơn → sai số giảm mạnh. Khi có thể thu thập/tạo thêm dữ liệu (kể cả bằng augmentation), đó thường là cách hiệu quả nhất.

## 7. Quy trình huấn luyện có hệ thống

Theo *A Recipe for Training Neural Networks* (Karpathy, 2019):
1. **Hiểu dữ liệu**: xem mẫu, phân phối nhãn, lỗi nhãn, trùng lặp, rò rỉ.
2. **Baseline đơn giản** + pipeline đánh giá đáng tin (cố định seed, metric đúng, so baseline ngây thơ).
3. **Overfit một batch nhỏ** (loss → ~0): xác nhận mô hình, loss, gradient hoạt động — nếu không làm được, có bug.
4. **Overfit toàn bộ train** với mô hình đủ lớn (đạt train loss thấp).
5. **Regularize**: thêm dữ liệu/augmentation → weight decay, dropout → early stopping; giảm kích thước nếu cần.
6. **Tuning**: learning rate trước tiên (LR range test), rồi kích thước, regularization (random search — chương 16).
7. **Vắt kiệt**: ensemble (chương 24), train lâu hơn với lịch học tốt.

| Triệu chứng | Nghi ngờ |
|---|---|
| Loss không giảm ngay từ đầu | LR quá nhỏ/lớn, bug dữ liệu (nhãn xáo trộn), quên `zero_grad`, sai loss (softmax 2 lần) |
| Loss = NaN | LR quá lớn, log(0), chia 0, dữ liệu có NaN → clipping, ổn định số |
| Train tốt, val tệ | overfitting → regularization, thêm dữ liệu; hoặc lệch phân phối train/val |
| Train và val đều tệ | underfitting → mô hình lớn hơn, train lâu hơn, LR, feature |
| Val tốt bất thường | rò rỉ dữ liệu (trùng mẫu giữa train/val) |
| Kết quả thay đổi mỗi lần đánh giá | quên `model.eval()` (dropout/BN) |

## 8. Template model: Multi-Layer Perceptron

| Mục | Nội dung |
|---|---|
| Problem | phân loại/hồi quy trên vector đặc trưng; khối xây dựng của mọi mạng sâu |
| Intuition | xếp chồng biến đổi tuyến tính + phi tuyến, học biểu diễn |
| Math | $\mathbf h_{l} = g(W_l\mathbf h_{l-1} + \mathbf b_l)$; loss CE/MSE; backprop |
| Algorithm | mini-batch SGD/Adam(W), lịch LR, early stopping |
| Complexity | $O(\sum_l n_l n_{l-1})$ mỗi mẫu cho cả forward và backward |
| Hyperparameters | số lớp, độ rộng, kích hoạt, LR, batch size, weight decay, dropout, số epoch |
| Assumptions | ít — nhưng không khai thác cấu trúc không gian/thời gian (→ CNN, RNN, Transformer) |
| Strengths | xấp xỉ hàm linh hoạt, mở rộng tốt với dữ liệu, GPU |
| Weaknesses | cần nhiều dữ liệu, nhiều siêu tham số, khó diễn giải, dữ liệu bảng thường thua boosting |
| Failure cases | vanishing gradient (sigmoid sâu), khởi tạo sai, LR sai, overfitting dữ liệu nhỏ |
| Production | chuẩn hoá đầu vào như lúc train, `eval()`, lưu `state_dict`, giám sát drift |
| Alternatives | gradient boosting (bảng), CNN (ảnh), RNN/Transformer (chuỗi) |

## 🧠 Kiểm tra nhanh

1. Dấu hiệu nào cho thấy mạng khớp cả nhiễu?
2. Weight decay làm hàm trơn hơn thế nào? Vì sao dùng AdamW?
3. Inverted dropout làm gì khi train và khi eval? Vì sao nhân $1/(1-p)$?
4. Early stopping cần gì? Vì sao nó là một dạng regularization?
5. "Overfit một batch nhỏ" dùng để kiểm tra điều gì?

## 📌 Tóm tắt

- Mạng lớn + ít dữ liệu → thuộc lòng nhiễu; đo bằng validation.
- Weight decay, dropout, nhiễu đầu vào/augmentation, early stopping — và trên hết là thêm dữ liệu.
- Huấn luyện có hệ thống: baseline → overfit batch → overfit train → regularize → tune.

## 📚 Tài liệu tham khảo

- Srivastava et al. (2014) — *Dropout: A Simple Way to Prevent Neural Networks from Overfitting*
- Bishop (1995) — *Training with Noise is Equivalent to Tikhonov Regularization*
- Prechelt (1998) — *Early Stopping — But When?*
- Karpathy (2019) — *A Recipe for Training Neural Networks* (karpathy.github.io)
- Goodfellow, Bengio & Courville (2016) — *Deep Learning*, ch. 7